In [ ]:

'''
class ModSiLU(nn.Module):
    """
    Complex ModSiLU：SiLU active on magnitude，keep phase unchanged
    SiLU(x) = x * sigmoid(x)
    """
    def forward(self, h_real, h_imag):
        magnitude = torch.sqrt(h_real**2 + h_imag**2 + 1e-8)
        scale = F.silu(magnitude) / (magnitude + 1e-8)
        return h_real * scale, h_imag * scale


class LinearTreeAttnLayer_SEP(nn.Module):
    def __init__(self, num_heads, hidden_dim, p=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.p = p

        # 核矩阵 B: [num_heads, head_dim, head_dim]
        # 如果 hidden_dim 不能被 num_heads 整除，head_dim 会被截断
        # 这相当于丢弃了多余的维度，但保留主要结构
        self.B = nn.Parameter(torch.randn(num_heads, self.head_dim, self.head_dim) * 0.01)
        #self.B_FUSE = nn.Parameter(torch.randn(num_heads,num_heads) * 0.01)
        self.gate = nn.Parameter(torch.randn(hidden_dim) * 0.01)
        # ★ 因果mask：只保留下三角（包括对角线）
        #self.register_buffer('causal_mask', None)
        

    def forward(self, x):
        """
        x: [B, T, 2, H]  0: real, 1: imag
        """
        B, T, _, H = x.shape
        d = self.head_dim
        #if self.causal_mask is None or self.causal_mask.size(0) != T:
        mask = torch.tril(torch.ones(T, T, device=x.device), diagonal=0).detach()  # [T, T]
        causal_mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, T, T]
        

        # 1. 只取实部计算注意力，重塑为多头
        z,zi = x[:, :, 0, :],x[:, :, 1, :]  # [B, T, H]
        z =  z.view(B, T, self.num_heads, d)  # [B, T, num_heads, d]
        zi = zi.view(B, T, self.num_heads, d)
        # 2. 计算多头注意力分数
        zB = torch.einsum('bthd, hdd -> bthd', z, self.B)  # [B, T, num_heads, d]
        attn_scores = torch.einsum('bthd, bHhd -> bthH', zB, z)  # [B, T, num_heads, T]
        attn_scores = torch.sigmoid(attn_scores)
        
        
        # 3. 提取次对角线（用于树结构惩罚）
        subdiag = torch.diagonal(attn_scores, offset=1, dim1=1, dim2=3)  # [B, num_heads, T-1] 
        subdiag = 1-subdiag
        assert not torch.any(subdiag < 0), f"subdiag has unsuitable values! "
        assert not torch.any(subdiag > 1), f"subdiag has unsuitable values! "
        # 4. 树结构惩罚（累积和版本）
        accu = torch.cumsum(torch.exp(subdiag), dim=-1)  # [B, num_heads, T-1]
        accu = torch.cat([torch.zeros(B, self.num_heads, 1, device=x.device), accu], dim=-1)  # [B, num_heads, T]
        #print(accu[0,0,:])
        C = accu.unsqueeze(-1) - accu.unsqueeze(-2)  # [B, num_heads, T, T]# [B, num_heads, T, T]
        C = C.masked_fill(causal_mask == 0, 0)
        assert not torch.any(C < 0), f"C has negative values! min: {C.min().item()}"
        C = torch.log(1+C+1e-6)
        assert not torch.any(C < 0), f"C has negative values after log! min: {C.min().item()}"
        # 5. 应用惩罚
        attn_scores = attn_scores.permute(0, 2, 1, 3)  # [B, num_heads, T, T]
        attn_scores = 1 + F.relu(attn_scores - self.p * C)
        
        uz  = torch.einsum('bntt , bntd -> bntd', attn_scores ,z.permute(0,2,1,3)).permute(0,2,1,3).reshape(B,T,-1)  
        uzi = torch.einsum('bntt , bntd -> bntd', attn_scores ,zi.permute(0,2,1,3)).permute(0,2,1,3).reshape(B,T,-1) 
        z   = torch.stack([uz,uzi],dim = 2)
     
        
        
        # 6. 跨头融合
        sim_maxsubdiag = C[:,:,-1,0] # [B, num_heads]
        zBF = torch.einsum('bh, hh -> bh', sim_maxsubdiag, self.B_FUSE)  # [B, num_heads]
        cross_attn = torch.einsum('bh, bH -> bhH', zBF, sim_maxsubdiag)  # [B, num_heads, num_heads]
        cross_attn = torch.sigmoid(cross_attn)
        cross_attn = cross_attn / (torch.sum(cross_attn, dim=-1, keepdim=True) + 1e-6)  # [B, num_heads, num_heads]

        # 更新 attn_scores
        attn_scores = torch.einsum('bhtT, bhh -> bhtT', attn_scores, cross_attn)  # [B, num_heads, T, T]

        # 因果mask + 归一化
        attn_scores = attn_scores.masked_fill(causal_mask == 0, 0)
        attn_scores = attn_scores / (torch.sum(attn_scores, dim=-1, keepdim=True) + 1e-6)  # [B, num_heads, T, T]

        # 取平均得到最终注意力
        attn_final = attn_scores.mean(dim=1)  # [B, T, T]
        
        # 7. 更新 x
        #x_flat = x.view(B, T, -1)
        #x_weighted = torch.matmul(attn_final, x_flat).view(B, T, 2, H)
        
        x_out = x * torch.sigmoid(self.gate) + F.silu(z)
        magnitude = torch.sqrt(x_out[:, :, 0, :]**2 + x_out[:, :, 1, :]**2 + 1e-8)
        x_out = x_out / magnitude.unsqueeze(2)

        return x_out
    
    
'''    
   
    
'''
Too complex hidden propagtion abandoned!

class ComplexPRLayer_strict(nn.Module):
    """
    complex state propagate layer（ComplexPRLayer_strict）
    mix real and image with  W = W_real + i W_imag 
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        # ★ complex weight matrix W = W_real + i W_imag
        self.W_real = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        self.W_imag = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)

        # ★ gate：input shape  2 * hidden_dim（concat real and imag）
        self.delta_proj = nn.Linear(2 * hidden_dim, 1)
        self.gamma_proj = nn.Linear(2 * hidden_dim, 1)
        self.skip_gate = nn.Parameter(torch.ones(hidden_dim) * 0.5)

        # active with magnitude
        self.mod_silu = ModSiLU()

        # initialization
        #nn.init.zeros_(self.delta_proj.weight)
        #nn.init.zeros_(self.delta_proj.bias)
        #nn.init.zeros_(self.gamma_proj.weight)
        #nn.init.zeros_(self.gamma_proj.bias)

    def forward(self, h_real_seq, h_imag_seq):
        """
        Args:
            h_real_seq, h_imag_seq: [B, T, H]
        Returns:
            h_real_out, h_imag_out: [B, T, H]
        """
        B, T, H = h_real_seq.shape

        # save input for skip connection 
        h_real_input = h_real_seq
        h_imag_input = h_imag_seq

        # initial of pre state of time 0
        h_real_prev = torch.zeros(B, H, device=h_real_seq.device)
        h_imag_prev = torch.zeros(B, H, device=h_real_seq.device)

        outputs_real, outputs_imag = [], []

        for t in range(T):
            h_real_t = h_real_seq[:, t, :]
            h_imag_t = h_imag_seq[:, t, :]

            # ★ combine real and image part
            h_combined = torch.cat([h_real_t, h_imag_t], dim=-1)  # [B, 2*H]

            # ★ complex matrix mul：W * h_prev
            W_h_real = torch.matmul(h_real_prev, self.W_real.T) - torch.matmul(h_imag_prev, self.W_imag.T)
            W_h_imag = torch.matmul(h_real_prev, self.W_imag.T) + torch.matmul(h_imag_prev, self.W_real.T)

            # ★ gate：
            delta_t = F.softplus(self.delta_proj(h_combined))
            alpha = torch.exp(-delta_t)
            
            # ★ gamma_t period gate (1 + sin(x)) / 2
            gamma_raw = self.gamma_proj(h_combined)  # [B, 1]
            gamma_t = (1 + torch.sin(gamma_raw)) / 2  # 范围 [0, 1]

            # ★ state update
            h_real_new = alpha * h_real_prev + gamma_t * W_h_real
            h_imag_new = alpha * h_imag_prev + gamma_t * W_h_imag

            outputs_real.append(h_real_new)
            outputs_imag.append(h_imag_new)

            # forward state
            h_real_prev = h_real_new
            h_imag_prev = h_imag_new

        # stack output
        h_real_recur = torch.stack(outputs_real, dim=1)
        h_imag_recur = torch.stack(outputs_imag, dim=1)

        # ★ activation of modsilu
        h_real_act, h_imag_act = self.mod_silu(h_real_recur, h_imag_recur)

        # ★ Skip ：gate * input + (1 - gate) * activated
        gate = torch.sigmoid(self.skip_gate).unsqueeze(0).unsqueeze(0)  # [1, 1, H]
        h_real_skip = h_real_input * gate + h_real_act 
        h_imag_skip = h_imag_input * gate + h_imag_act 

        # ★ complex normalization
        magnitude = torch.sqrt(h_real_skip**2 + h_imag_skip**2 + 1e-8)
        h_real_out = h_real_skip / magnitude
        h_imag_out = h_imag_skip / magnitude

        return h_real_out, h_imag_out
'''
    
    